In [2]:
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from pathlib import Path
from PIL import Image

# ─── Paths ────────────────────────────────────────────────────────────────────
DATASET_PATH    = Path(r"C:\Users\jessi\OneDrive\Desktop\aiims2")
CHECKPOINT_PATH = DATASET_PATH / "UltraSam.pth"
manifest        = pd.read_csv(DATASET_PATH / 'processed_manifest.csv')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")
print(f"Total cases:  {len(manifest)}")

# ─── Download checkpoint if missing ───────────────────────────────────────────
if not CHECKPOINT_PATH.exists():
    import urllib.request
    print("Downloading UltraSam checkpoint (~375 MB)…")
    urllib.request.urlretrieve(
        "https://s3.unistra.fr/camma_public/github/ultrasam/UltraSam.pth",
        CHECKPOINT_PATH,
    )
    print("Download complete.")

# ─── Load UltraSam image encoder ──────────────────────────────────────────────
from segment_anything import sam_model_registry

sam = sam_model_registry["vit_b"](checkpoint=None)

# Load and unwrap the MMSeg-style checkpoint
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
state_dict = checkpoint['state_dict']

# Remap MMSeg backbone key names to SAM ImageEncoderViT key names
def remap_key(k):
    k = re.sub(r'^layers\.(\d+)\.ln1',           r'blocks.\1.norm1',     k)
    k = re.sub(r'^layers\.(\d+)\.ln2',           r'blocks.\1.norm2',     k)
    k = re.sub(r'^layers\.(\d+)\.attn',          r'blocks.\1.attn',      k)
    k = re.sub(r'^layers\.(\d+)\.ffn\.layers\.0\.0', r'blocks.\1.mlp.lin1', k)
    k = re.sub(r'^layers\.(\d+)\.ffn\.layers\.1',    r'blocks.\1.mlp.lin2', k)
    k = k.replace('patch_embed.projection', 'patch_embed.proj')
    k = k.replace('channel_reduction', 'neck')
    return k

encoder_state = {
    remap_key(k.replace("backbone.", "")): v
    for k, v in state_dict.items()
    if k.startswith("backbone.")
}
sam.image_encoder.load_state_dict(encoder_state, strict=True)

encoder = sam.image_encoder.to(DEVICE)
encoder.eval()
print("UltraSam image encoder loaded — output shape: (256, 64, 64) → pooled to (256,)")

# ─── Preprocessing ────────────────────────────────────────────────────────────
SAM_MEAN = [123.675, 116.28, 103.53]
SAM_STD  = [58.395,  57.12,  57.375]

transform = transforms.Compose([
    transforms.Resize((1024, 1024)),
    transforms.ToTensor(),
    transforms.Lambda(lambda t: t * 255.0),
    transforms.Normalize(mean=SAM_MEAN, std=SAM_STD),
])

def to_rgb_tensor(crop_array: np.ndarray) -> torch.Tensor:
    arr = crop_array.copy()
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8) * 255
    arr = arr.astype(np.uint8)
    pil_img = Image.fromarray(arr).convert("RGB")
    return transform(pil_img).unsqueeze(0).to(DEVICE)

# ─── Spatial pooling: (1,256,64,64) → (256,) ─────────────────────────────────
gap = nn.AdaptiveAvgPool2d(1)

# ─── Bounding-box helpers ─────────────────────────────────────────────────────
def get_bounding_box(mask, padding=10):
    rows = np.where(mask.sum(axis=1) > 0)[0]
    cols = np.where(mask.sum(axis=0) > 0)[0]
    if len(rows) == 0 or len(cols) == 0:
        return None
    h, w = mask.shape
    r_min = max(0, rows.min() - padding)
    r_max = min(h, rows.max() + padding)
    c_min = max(0, cols.min() - padding)
    c_max = min(w, cols.max() + padding)
    return r_min, r_max, c_min, c_max

def extract_crops(image, masks):
    crops = {}
    bbox = get_bounding_box(masks['tumor_mask'], padding=5)
    crops['tumor'] = image[bbox[0]:bbox[1], bbox[2]:bbox[3]] if bbox else image

    combined_mask = np.clip(masks['tumor_mask'] + masks['ring_15px'], 0, 1)
    bbox = get_bounding_box(combined_mask, padding=10)
    crops['perilesion'] = image[bbox[0]:bbox[1], bbox[2]:bbox[3]] if bbox else image

    bbox = get_bounding_box(masks['posterior_mask'], padding=0)
    crops['posterior'] = image[bbox[0]:bbox[1], bbox[2]:bbox[3]] if bbox else image

    crops['whole'] = image
    return crops

# ─── Embedding extraction ──────────────────────────────────────────────────────
def get_embedding(crop_array: np.ndarray) -> np.ndarray:
    tensor = to_rgb_tensor(crop_array)
    with torch.no_grad():
        feat = encoder(tensor)
        emb  = gap(feat).squeeze()
    return emb.cpu().numpy()

# ─── Quick sanity check ───────────────────────────────────────────────────────
sample_row = manifest.iloc[0]
sample     = np.load(sample_row['npz_path'])
image      = sample['image']
masks = {k: sample[k] for k in
         ['tumor_mask', 'ring_5px', 'ring_15px', 'posterior_mask',
          'skin_mask', 'background_mask']}

crops = extract_crops(image, masks)
print("Crop shapes:")
for name, crop in crops.items():
    print(f"  {name}: {crop.shape}")

emb = get_embedding(crops['tumor'])
print(f"\nTumor embedding shape : {emb.shape}")
print(f"First 5 values        : {emb[:5]}")

# ─── Main loop ────────────────────────────────────────────────────────────────
CROP_NAMES  = ['tumor', 'perilesion', 'posterior', 'whole']
all_records = []

for idx, row in manifest.iterrows():
    if idx % 50 == 0:
        print(f"Processing {idx}/{len(manifest)}…")
    try:
        sample = np.load(row['npz_path'])
        image  = sample['image']
        masks  = {k: sample[k] for k in
                  ['tumor_mask', 'ring_5px', 'ring_15px', 'posterior_mask',
                   'skin_mask', 'background_mask']}
        crops  = extract_crops(image, masks)

        record = {'case_id': row['case_id'], 'label': row['label']}
        for crop_name in CROP_NAMES:
            emb = get_embedding(crops[crop_name])
            for i, val in enumerate(emb):
                record[f'{crop_name}_{i}'] = float(val)

        all_records.append(record)
    except Exception as e:
        print(f"ERROR on {row['case_id']}: {e}")

print(f"\nDone! Total records: {len(all_records)}")

# ─── Save ─────────────────────────────────────────────────────────────────────
df_deep = pd.DataFrame(all_records)
df_deep.to_csv(DATASET_PATH / 'features_deep_ultrasam.csv', index=False)
print(f"Saved features_deep_ultrasam.csv")
print(f"Shape: {df_deep.shape}")
print(f"Columns: case_id, label + {len(df_deep.columns)-2} embedding dims")

# ─── Merge with texture features ──────────────────────────────────────────────
df_texture      = pd.read_csv(DATASET_PATH / 'features_texture.csv')
df_texture_wide = df_texture.pivot_table(
    index=['case_id', 'label'],
    columns='region',
    values=[c for c in df_texture.columns if c not in ['case_id','label','region']],
    aggfunc='first',
).reset_index()
df_texture_wide.columns = [
    f"{col[0]}_{col[1]}" if col[1] else col[0]
    for col in df_texture_wide.columns
]
df_combined = df_texture_wide.merge(df_deep, on=['case_id','label'], how='inner')
df_combined.to_csv(DATASET_PATH / 'features_combined_ultrasam.csv', index=False)

print(f"features_texture wide shape : {df_texture_wide.shape}")
print(f"features_deep shape         : {df_deep.shape}")
print(f"features_combined shape     : {df_combined.shape}")

Using device: cuda
Total cases:  780
UltraSam image encoder loaded — output shape: (256, 64, 64) → pooled to (256,)
Crop shapes:
  tumor: (23, 32)
  perilesion: (63, 72)
  posterior: (168, 255)
  whole: (256, 256)

Tumor embedding shape : (256,)
First 5 values        : [-0.00344687 -0.12358747  0.04979782 -0.0145057   0.04046247]
Processing 0/780…
Processing 50/780…
Processing 100/780…
Processing 150/780…
Processing 200/780…
Processing 250/780…
Processing 300/780…
Processing 350/780…
Processing 400/780…
Processing 450/780…
Processing 500/780…
Processing 550/780…
Processing 600/780…
Processing 650/780…
Processing 700/780…
Processing 750/780…

Done! Total records: 780
Saved features_deep_ultrasam.csv
Shape: (780, 1026)
Columns: case_id, label + 1024 embedding dims


KeyError: 'region'

In [4]:
import pandas as pd
from pathlib import Path

DATASET_PATH = Path(r"C:\Users\jessi\OneDrive\Desktop\aiims2")

df_texture = pd.read_csv(DATASET_PATH / 'features_texture.csv')
df_deep    = pd.read_csv(DATASET_PATH / 'features_deep_ultrasam.csv')

df_combined = df_texture.merge(df_deep, on=['case_id', 'label'], how='inner')
df_combined.to_csv(DATASET_PATH / 'features_combined_ultrasam.csv', index=False)

print(f"features_texture shape  : {df_texture.shape}")
print(f"features_deep shape     : {df_deep.shape}")
print(f"features_combined shape : {df_combined.shape}")

features_texture shape  : (780, 134)
features_deep shape     : (780, 1026)
features_combined shape : (780, 1158)


In [3]:
import sys
!{sys.executable} -m pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 --upgrade --force-reinstall

Looking in indexes: https://download.pytorch.org/whl/cu121


ERROR: Could not find a version that satisfies the requirement torch (from versions: none)

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\jessi\myenv\Scripts\python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for torch


In [6]:
pip install mmengine


   ----- ---------------------------------- 1/7 [yapf]
   ----- ---------------------------------- 1/7 [yapf]
   ----- ---------------------------------- 1/7 [yapf]
   ----- ---------------------------------- 1/7 [yapf]
   ---------------------- ----------------- 4/7 [markdown-it-py]
   ---------------------- ----------------- 4/7 [markdown-it-py]
   ---------------------- ----------------- 4/7 [markdown-it-py]
   ---------------------- ----------------- 4/7 [markdown-it-py]
   ---------------------------- ----------- 5/7 [rich]
   ---------------------------- ----------- 5/7 [rich]
   ---------------------------- ----------- 5/7 [rich]
   ---------------------------- ----------- 5/7 [rich]
   ---------------------------- ----------- 5/7 [rich]
   ---------------------------------- ----- 6/7 [mmengine]
   ---------------------------------- ----- 6/7 [mmengine]
   ---------------------------------- ----- 6/7 [mmengine]
   ---------------------------------- ----- 6/7 [mmengine]
   -----


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\jessi\myenv\Scripts\python.exe -m pip install --upgrade pip


In [11]:
print(checkpoint["meta"])

{'epoch': 0, 'iter': 30000, 'cfg': "N_POINTS = 1\ncustom_hooks = [\n    dict(type='MonkeyPatchHook'),\n]\ncustom_imports = dict(\n    allow_failed_imports=False,\n    imports=[\n        'mmpretrain.models',\n        'endosam.datasets.transforms.custom_pipeline',\n        'endosam.datasets.transforms.point_formatting',\n        'endosam.visualization.point_visualization',\n        'endosam.models.detectors.SAM',\n        'endosam.models.backbones.vit_sam',\n        'endosam.models.dense_heads.sam_mask_decoder',\n        'endosam.models.dense_heads.sam_mask_class_decoder',\n        'endosam.datasets.evaluation.LabelMetric',\n        'endosam.models.utils.sam_layers',\n        'endosam.models.task_modules.prior_generators.prompt_encoder',\n        'endosam.models.task_modules.prior_generators.label_encoder',\n        'endosam.hooks.MonkeyPatchHook',\n    ])\ndata_root = 'UltraSAM_DATA/UltraSAM'\ndataset_type = 'CocoDataset'\ndefault_hooks = dict(\n    checkpoint=dict(\n        by_epoch=Fa

In [13]:
print(list(state_dict.keys())[:20])  # See what keys actually look like

['meta', 'state_dict', 'message_hub', 'param_schedulers']


In [17]:
encoder_state = {
    k.replace("backbone.", ""): v
    for k, v in state_dict.items()
    if k.startswith("backbone.")
}

sam.image_encoder.load_state_dict(encoder_state, strict=True)

RuntimeError: Error(s) in loading state_dict for ImageEncoderViT:
	Missing key(s) in state_dict: "patch_embed.proj.weight", "patch_embed.proj.bias", "blocks.0.norm1.weight", "blocks.0.norm1.bias", "blocks.0.attn.rel_pos_h", "blocks.0.attn.rel_pos_w", "blocks.0.attn.qkv.weight", "blocks.0.attn.qkv.bias", "blocks.0.attn.proj.weight", "blocks.0.attn.proj.bias", "blocks.0.norm2.weight", "blocks.0.norm2.bias", "blocks.0.mlp.lin1.weight", "blocks.0.mlp.lin1.bias", "blocks.0.mlp.lin2.weight", "blocks.0.mlp.lin2.bias", "blocks.1.norm1.weight", "blocks.1.norm1.bias", "blocks.1.attn.rel_pos_h", "blocks.1.attn.rel_pos_w", "blocks.1.attn.qkv.weight", "blocks.1.attn.qkv.bias", "blocks.1.attn.proj.weight", "blocks.1.attn.proj.bias", "blocks.1.norm2.weight", "blocks.1.norm2.bias", "blocks.1.mlp.lin1.weight", "blocks.1.mlp.lin1.bias", "blocks.1.mlp.lin2.weight", "blocks.1.mlp.lin2.bias", "blocks.2.norm1.weight", "blocks.2.norm1.bias", "blocks.2.attn.rel_pos_h", "blocks.2.attn.rel_pos_w", "blocks.2.att

In [16]:
def remap_key(k):
    import re
    # layers.N.ln1 -> blocks.N.norm1
    k = re.sub(r'^layers\.(\d+)\.ln1', r'blocks.\1.norm1', k)
    # layers.N.ln2 -> blocks.N.norm2
    k = re.sub(r'^layers\.(\d+)\.ln2', r'blocks.\1.norm2', k)
    # layers.N.attn -> blocks.N.attn
    k = re.sub(r'^layers\.(\d+)\.attn', r'blocks.\1.attn', k)
    # layers.N.ffn.layers.0.0 -> blocks.N.mlp.lin1
    k = re.sub(r'^layers\.(\d+)\.ffn\.layers\.0\.0', r'blocks.\1.mlp.lin1', k)
    # layers.N.ffn.layers.1 -> blocks.N.mlp.lin2
    k = re.sub(r'^layers\.(\d+)\.ffn\.layers\.1', r'blocks.\1.mlp.lin2', k)
    # patch_embed.projection -> patch_embed.proj
    k = k.replace('patch_embed.projection', 'patch_embed.proj')
    # channel_reduction -> neck
    k = k.replace('channel_reduction', 'neck')
    return k

encoder_state = {
    remap_key(k.replace("backbone.", "")): v
    for k, v in state_dict.items()
    if k.startswith("backbone.")
}

sam.image_encoder.load_state_dict(encoder_state, strict=True)

<All keys matched successfully>

In [18]:
encoder = sam.image_encoder.to(DEVICE)
encoder.eval()

ImageEncoderViT(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
  )
  (blocks): ModuleList(
    (0-11): 12 x Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (proj): Linear(in_features=768, out_features=768, bias=True)
      )
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): MLPBlock(
        (lin1): Linear(in_features=768, out_features=3072, bias=True)
        (lin2): Linear(in_features=3072, out_features=768, bias=True)
        (act): GELU(approximate='none')
      )
    )
  )
  (neck): Sequential(
    (0): Conv2d(768, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (1): LayerNorm2d()
    (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (3): LayerNorm2d()
  )
)